# Harvest API-IL papers from PubMed (run in Colab)
This collects titles + abstracts of papers on drug-based ionic liquids, so every new API-IL pair can be traced to a DOI.
It does **not** decide anything - it only downloads. Upload the resulting `api_il_abstracts.csv` back to Claude for extraction.
Runtime: roughly 10-20 minutes.

In [ ]:
!pip install -q biopython

In [ ]:
import re, time, pandas as pd
from Bio import Entrez
Entrez.email = "YOUR_EMAIL_HERE"     # NCBI asks for a contact email; put your own
IL = '("ionic liquid*"[tiab] OR "protic salt*"[tiab] OR "oily salt*"[tiab] OR "liquid salt*"[tiab])'
CORE = [
 '("API-IL"[tiab] OR "API-ILs"[tiab] OR "API-based ionic liquid*"[tiab] OR "drug-based ionic liquid*"[tiab] OR "pharmaceutically active ionic liquid*"[tiab] OR "dual active ionic liquid*"[tiab] OR "dual-active ionic liquid*"[tiab])',
 '("ionic liquid*"[tiab] AND ("active pharmaceutical ingredient*"[tiab] OR "drug salt*"[tiab] OR counterion*[tiab] OR "counter-ion*"[tiab]))',
]
DRUGS = """ibuprofen naproxen ketoprofen flurbiprofen diclofenac indomethacin "mefenamic acid" "tolfenamic acid" "flufenamic acid"
"meclofenamic acid" "niflumic acid" "salicylic acid" salicylate "acetylsalicylic acid" aspirin piroxicam meloxicam loxoprofen etodolac
sulindac ketorolac lidocaine bupivacaine ropivacaine tetracaine procaine prilocaine benzocaine mepivacaine ampicillin amoxicillin
penicillin ciprofloxacin norfloxacin ofloxacin levofloxacin moxifloxacin "nalidixic acid" sulfadiazine sulfamethoxazole sulfacetamide
sulfathiazole tetracycline doxycycline acyclovir "valproic acid" valproate gabapentin pregabalin tramadol amitriptyline imipramine
chlorpromazine fluoxetine sertraline propranolol atenolol metoprolol carvedilol amlodipine verapamil diltiazem metformin glibenclamide
gliclazide tolbutamide ranitidine cimetidine famotidine diphenhydramine cetirizine fexofenadine itraconazole miconazole ketoconazole
fluconazole mebendazole albendazole praziquantel primaquine chloroquine lumefantrine artesunate isoniazid rifampicin pyrazinamide
ethambutol methotrexate "5-fluorouracil" "folic acid" "nicotinic acid" theophylline edaravone "tranexamic acid" baclofen levodopa
memantine rivastigmine donepezil pyridostigmine neostigmine mepenzolate glycopyrronium benzethonium cetylpyridinium chlorhexidine
benzalkonium miltefosine "ursolic acid" "betulinic acid" "oleanolic acid" "ferulic acid" "caffeic acid" "gallic acid" furosemide
probenecid warfarin docusate saccharinate acesulfame choline""".replace("\n"," ")
DRUGS = re.findall(r'"[^"]+"|\S+', DRUGS)
QUERIES = CORE + [f'{d}[tiab] AND {IL}' for d in DRUGS]
print(len(QUERIES), 'queries')

In [ ]:
def search(term, retmax=5000):
    for attempt in range(3):
        try:
            h = Entrez.esearch(db="pubmed", term=term, retmax=retmax); r = Entrez.read(h); h.close()
            return r["IdList"]
        except Exception as e:
            time.sleep(3)
    return []

def fetch(pmids):
    rows = []
    for i in range(0, len(pmids), 200):
        chunk = pmids[i:i+200]
        for attempt in range(3):
            try:
                h = Entrez.efetch(db="pubmed", id=",".join(chunk), retmode="xml"); recs = Entrez.read(h); h.close(); break
            except Exception:
                time.sleep(3); recs = {"PubmedArticle": []}
        for a in recs["PubmedArticle"]:
            c = a["MedlineCitation"]; art = c["Article"]
            abst = " ".join(str(x) for x in art.get("Abstract", {}).get("AbstractText", []))
            doi = next((str(x) for x in a["PubmedData"].get("ArticleIdList", []) if x.attributes.get("IdType") == "doi"), "")
            yr = art.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {}).get("Year", "")
            rows.append(dict(pmid=str(c["PMID"]), doi=doi, year=yr, journal=art.get("Journal", {}).get("Title", ""),
                             title=str(art.get("ArticleTitle", "")), abstract=abst))
        time.sleep(0.4)
    return rows

In [ ]:
hits = {}
for q in QUERIES:
    for p in search(q):
        hits.setdefault(p, set()).add(q[:60])
    time.sleep(0.4)
print(len(hits), 'unique papers')
rows = fetch(list(hits))
df = pd.DataFrame(rows)
df["found_by"] = df.pmid.map(lambda p: " | ".join(sorted(hits.get(p, []))))
# helper only: bracket names like [Chol][Ibu] and "...ium ...ate" phrases, to speed up reading
pat = re.compile(r"\[[^\]]{1,40}\]\s*\[[^\]]{1,40}\]|\b[\w-]+ium\b(?:\s+[\w-]+){0,2}?\s+[\w-]+(?:ate|ide|inate|enate)\b", re.I)
df["candidate_phrases"] = (df.title + " " + df.abstract).map(lambda t: " ; ".join(sorted(set(m.group(0) for m in pat.finditer(t))))[:1000])
df.to_csv("api_il_abstracts.csv", index=False)
print(df.shape, "| with candidate phrases:", (df.candidate_phrases != "").sum())

In [ ]:
from google.colab import files
files.download("api_il_abstracts.csv")